# iDILI-Predict Preprocessing Example

Example workflow of the preprocessing of the CellProfiler output

In this example we use 72.3 cells treated with the mechanistic standards for DILI. Put the link for the example measuremnts and metadata zip here



In [1]:
import os
import re

import numpy as np
import pandas as pd

from idili.preprocess import merge_files, agg_norm, dmso_norm, long_to_wide

df = pd.read_csv("example_data/data.csv")
columns = pd.read_csv('example_data/column_types.csv', sep=',')

data_cols = columns.loc[columns['column_type']=='feature', 'column_name'].unique().tolist()
meta_cols = columns.loc[columns['column_type']=='metadata', 'column_name'].unique().tolist()

## Aggregation and Plate-based Normalization

Raw CellProfiler output was processed using pycytominer (v1.3.0). Cell-level CellProfiler measurements are aggregated to well-level using median values, then normalized per plate using MAD robustization against DMSO controls to correct for plate effects.

In [3]:
df_norm = agg_norm(df, data_cols=data_cols, meta_cols=meta_cols)
print(df.shape)
print(df_norm.shape)
df_norm.head()

(82763, 519)
(115, 518)


/home/ben/anaconda3/envs/pycytominer/lib/python3.13/site-packages/pycytominer/aggregate.py:116: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  population_df = population_df.median().reset_index()
/home/ben/anaconda3/envs/pycytominer/lib/python3.13/site-packages/pycytominer/aggregate.py:116: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  population_df = population_df.median().reset_index()


,ImageNumber,Metadata_Compound,Metadata_Control,Metadata_Concentration,Metadata_PlateID,Metadata_WellID,Cell_AreaShape_Area,Cell_AreaShape_BoundingBoxArea,Cell_AreaShape_BoundingBoxMaximum_X,Cell_AreaShape_BoundingBoxMaximum_Y,...,Nucleus_Intensity_StdIntensity_CellMask,Nucleus_Intensity_StdIntensity_DNA,Nucleus_Intensity_StdIntensity_MT,Nucleus_Intensity_StdIntensity_enhLipids,Nucleus_Intensity_StdIntensity_lipid,Nucleus_Intensity_UpperQuartileIntensity_CellMask,Nucleus_Intensity_UpperQuartileIntensity_DNA,Nucleus_Intensity_UpperQuartileIntensity_MT,Nucleus_Intensity_UpperQuartileIntensity_enhLipids,Nucleus_Intensity_UpperQuartileIntensity_lipid
0,7,DMSO,NC,0.0,Plate1,A-07,4.636364,5.148232,-2.094772,0.473327,...,15.770072,10.775207,0.128910,4.968906,6.299040,27.935503,11.791098,0.563989,4.721435,7.440476
1,18,DMSO,NC,0.0,Plate1,A-18,12.029938,14.732098,-4.007017,-4.332420,...,46.665331,108.948663,0.320454,10.259818,15.507444,100.004635,118.027557,0.961508,8.318719,20.276879
2,42,Indinavir,MS,3.5,Plate1,B-18,9.974564,12.220858,0.529040,2.484966,...,67.473653,91.635911,0.674491,11.153232,17.010539,163.725388,118.169117,1.971809,9.442871,28.412923
3,47,Indinavir,MS,11.0,Plate1,B-23,11.375196,13.878505,-1.244885,3.180165,...,69.880927,99.195958,0.580387,11.111596,17.481974,142.003067,113.714146,1.522627,8.655966,24.302746
4,60,Indinavir,MS,33.5,Plate1,C-12,12.809248,15.730496,-3.124332,0.056208,...,66.723386,64.397080,0.966430,12.664343,17.966562,144.117577,88.383271,2.006251,8.993210,25.967896


In [4]:
df_norm.to_csv("example_data/aggregated_normalized.csv", index=False)

## DMSO Earth Mover's Distance

For BMDExpress3 input, three representative DMSO replicates per donor are selected using Wasserstein distance (Earth Mover's Distance) to identify the most concordant medoid observations among all DMSO samples.

In [6]:
df_dmso = dmso_norm(df_norm, meta_cols=meta_cols, data_cols=data_cols)
print("Normalized dataframe shape:", df_norm.shape)
print("EMD dataframe:", df_dmso.shape)
df_dmso.head()

Normalized dataframe shape: (115, 518)
EMD dataframe: (76, 518)


,ImageNumber,Metadata_Compound,Metadata_Control,Metadata_Concentration,Metadata_PlateID,Metadata_WellID,Cell_AreaShape_Area,Cell_AreaShape_BoundingBoxArea,Cell_AreaShape_BoundingBoxMaximum_X,Cell_AreaShape_BoundingBoxMaximum_Y,...,Nucleus_Intensity_StdIntensity_CellMask,Nucleus_Intensity_StdIntensity_DNA,Nucleus_Intensity_StdIntensity_MT,Nucleus_Intensity_StdIntensity_enhLipids,Nucleus_Intensity_StdIntensity_lipid,Nucleus_Intensity_UpperQuartileIntensity_CellMask,Nucleus_Intensity_UpperQuartileIntensity_DNA,Nucleus_Intensity_UpperQuartileIntensity_MT,Nucleus_Intensity_UpperQuartileIntensity_enhLipids,Nucleus_Intensity_UpperQuartileIntensity_lipid
2,42,Indinavir,MS,3.5,Plate1,B-18,9.974564,12.220858,0.529040,2.484966,...,67.473653,91.635911,0.674491,11.153232,17.010539,163.725388,118.169117,1.971809,9.442871,28.412923
3,47,Indinavir,MS,11.0,Plate1,B-23,11.375196,13.878505,-1.244885,3.180165,...,69.880927,99.195958,0.580387,11.111596,17.481974,142.003067,113.714146,1.522627,8.655966,24.302746
4,60,Indinavir,MS,33.5,Plate1,C-12,12.809248,15.730496,-3.124332,0.056208,...,66.723386,64.397080,0.966430,12.664343,17.966562,144.117577,88.383271,2.006251,8.993210,25.967896
5,61,Indinavir,MS,11.0,Plate1,C-13,12.962679,16.118233,0.620303,0.985112,...,60.144488,58.248850,2.109092,11.883461,16.727107,135.998855,80.739045,3.699653,8.768375,24.365979
9,100,Indinavir,MS,11.0,Plate1,E-04,11.628889,14.503457,0.634563,2.076722,...,56.101025,61.994074,3.027308,11.702976,15.132525,134.194691,89.798872,5.588227,8.543553,24.618914


## Long-to-Wide Transformation

Re-embed concentration into the feature columns. This reduces the number of observations, but increases number of features per observations.

Resulting output includes the long-to-wide dataframe, the new data columns that include dose and metadata columns

Inline output is current compound being processed, original dataframe shape, and long-to-wide dataframe shape

In [8]:
df_wide, wide_data, wide_meta = long_to_wide(df_dmso, data_cols=data_cols, meta_cols=meta_cols)
print("Input Dataframe shape:", df_dmso.shape)
print("Long-to-Wide Dataframe shape:", df_wide.shape)

Indinavir
Clozapine
Troglitazone
Chloroquine
Tamoxifen
(18, 2560) (88, 520)
Input Dataframe shape: (76, 518)
Long-to-Wide Dataframe shape: (18, 2568)


In [10]:
import pickle

In [12]:
df_dmso.to_csv('example_data/df_emd_dmso.csv', index=False)
df_wide.to_csv("example_data/df_wide.csv", index=False)

with open('example_data/wide_data.pkl', 'wb') as f:
    pickle.dump(wide_data, f)
with open('example_data/wide_meta.pkl', 'wb') as f:
    pickle.dump(wide_meta, f)

In [13]:
wide_cols = {
    'column_name':wide_data+wide_meta,
    'column_type':['feature' for _ in wide_data]+['metadata' for _ in wide_meta]
}
wide_cols = pd.DataFrame(data=wide_cols)

In [15]:
wide_cols.to_csv('example_data/wide_columns.csv', index=False)